# DCGAN 이미지 생성 — Cats & Dogs (PyTorch) — Colab

**DCGAN**(Deep Convolutional GAN)으로 고양이·강아지 이미지를 **생성**하는 기본 예제입니다.

- 핵심 흐름: **Generator(노이즈→이미지) vs Discriminator(진짜/가짜 판별)** 를 적대적으로 학습.
- [Dogs vs Cats](https://www.kaggle.com/c/dogs-vs-cats-redux-kernels-edition) 의 실제 사진을 64×64 로 학습해 새 이미지를 생성합니다.

> 📌 GAN 대표 대회 *Generative Dog Images* 는 2019년 종료되어 가입/다운로드가 막혀 있어, 동일한 DCGAN 코드를 **가입 가능한 Cats & Dogs** 이미지에 적용했습니다.

- 실행 시 **Kaggle API로 데이터를 직접 다운로드**하며, 토큰이 **노트북에 내장**되어 바로 실행됩니다.
- 학습 후 생성 샘플을 시각화하고, 생성 이미지를 `/content` 에 저장합니다.

> ⚙️ **GPU 권장**: 런타임 → 런타임 유형 변경 → GPU (GAN 학습은 CPU에서 매우 느림).  
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "torchvision", "numpy", "pillow", "matplotlib"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 이미지 다운로드 (Cats & Dogs)

> 약 800MB라 수 분 걸릴 수 있습니다.

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_file("dogs-vs-cats-redux-kernels-edition", "train.zip", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
IMG_DIR = os.path.join(WORK_DIR, "train")
print("images:", len(glob.glob(os.path.join(IMG_DIR, "*.jpg"))))

## 2. Dataset / DataLoader

고양이·강아지 이미지를 64×64 로 리사이즈하고 `[-1, 1]` 로 정규화합니다 (Generator 의 tanh 출력 범위와 맞춤).  
> 기본은 **cat & dog 전체**를 사용합니다. 특정 종류만 보려면 `only` 를 `"cat"` 또는 `"dog"` 로 바꾸세요.

In [ ]:
import glob, torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMG_SIZE = 64
tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

class ImgDataset(Dataset):
    def __init__(self, files, tf): self.files = files; self.tf = tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i): return self.tf(Image.open(self.files[i]).convert("RGB"))

only = "all"   # "all"(고양이+강아지) | "cat" | "dog"
files = glob.glob(os.path.join(IMG_DIR, "*.jpg"))
if only in ("cat", "dog"):
    files = [f for f in files if only in os.path.basename(f)]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loader = DataLoader(ImgDataset(files, tf), batch_size=128, shuffle=True, num_workers=2, drop_last=True)
print("device:", device, "| training images:", len(files), f"(only={only})")

## 3. Generator / Discriminator 정의 (DCGAN)

In [ ]:
import torch.nn as nn

NZ = 100   # 잠재 벡터 차원
NGF = 64   # generator feature
NDF = 64   # discriminator feature

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(NZ, NGF*8, 4, 1, 0, bias=False), nn.BatchNorm2d(NGF*8), nn.ReLU(True),   # 4x4
            nn.ConvTranspose2d(NGF*8, NGF*4, 4, 2, 1, bias=False), nn.BatchNorm2d(NGF*4), nn.ReLU(True), # 8x8
            nn.ConvTranspose2d(NGF*4, NGF*2, 4, 2, 1, bias=False), nn.BatchNorm2d(NGF*2), nn.ReLU(True), # 16x16
            nn.ConvTranspose2d(NGF*2, NGF, 4, 2, 1, bias=False), nn.BatchNorm2d(NGF), nn.ReLU(True),     # 32x32
            nn.ConvTranspose2d(NGF, 3, 4, 2, 1, bias=False), nn.Tanh(),                                 # 64x64
        )
    def forward(self, x): return self.net(x)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, NDF, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),                            # 32x32
            nn.Conv2d(NDF, NDF*2, 4, 2, 1, bias=False), nn.BatchNorm2d(NDF*2), nn.LeakyReLU(0.2, True),  # 16x16
            nn.Conv2d(NDF*2, NDF*4, 4, 2, 1, bias=False), nn.BatchNorm2d(NDF*4), nn.LeakyReLU(0.2, True),# 8x8
            nn.Conv2d(NDF*4, NDF*8, 4, 2, 1, bias=False), nn.BatchNorm2d(NDF*8), nn.LeakyReLU(0.2, True),# 4x4
            nn.Conv2d(NDF*8, 1, 4, 1, 0, bias=False), nn.Sigmoid(),                                     # 1
        )
    def forward(self, x): return self.net(x).view(-1)

def weights_init(m):
    cn = m.__class__.__name__
    if "Conv" in cn: nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in cn: nn.init.normal_(m.weight.data, 1.0, 0.02); nn.init.constant_(m.bias.data, 0)

netG = Generator().to(device).apply(weights_init)
netD = Discriminator().to(device).apply(weights_init)
print("Generator / Discriminator 준비 완료")

## 4. 적대적 학습

Discriminator 는 진짜/가짜를 구분하도록, Generator 는 Discriminator 를 속이도록 번갈아 학습합니다.

In [ ]:
import torch.optim as optim

criterion = nn.BCELoss()
optD = optim.Adam(netD.parameters(), lr=2e-4, betas=(0.5, 0.999))
optG = optim.Adam(netG.parameters(), lr=2e-4, betas=(0.5, 0.999))
fixed_noise = torch.randn(64, NZ, 1, 1, device=device)
REAL, FAKE = 1.0, 0.0

EPOCHS = 30  # Colab GPU 기준. 더 오래 학습할수록 품질 향상
for epoch in range(1, EPOCHS + 1):
    for real in loader:
        real = real.to(device); bs = real.size(0)
        # --- Discriminator ---
        netD.zero_grad()
        lossD_real = criterion(netD(real), torch.full((bs,), REAL, device=device))
        noise = torch.randn(bs, NZ, 1, 1, device=device)
        fake = netG(noise)
        lossD_fake = criterion(netD(fake.detach()), torch.full((bs,), FAKE, device=device))
        (lossD_real + lossD_fake).backward(); optD.step()
        # --- Generator ---
        netG.zero_grad()
        lossG = criterion(netD(fake), torch.full((bs,), REAL, device=device))  # 진짜로 속이기
        lossG.backward(); optG.step()
    print(f"epoch {epoch:2d} | lossD {(lossD_real+lossD_fake).item():.3f} | lossG {lossG.item():.3f}")

## 5. 생성 샘플 시각화

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

netG.eval()
with torch.no_grad():
    gen = netG(fixed_noise).cpu()
grid = make_grid(gen, nrow=8, normalize=True)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0)); plt.axis("off"); plt.title("Generated Cats & Dogs")
plt.show()

## 6. 생성 이미지 저장 & 내려받기

In [ ]:
from torchvision.utils import save_image
import zipfile, glob

OUT_DIR = os.path.join(WORK_DIR, "generated"); os.makedirs(OUT_DIR, exist_ok=True)
N_GEN = 200  # 생성 이미지 수
netG.eval()
with torch.no_grad():
    for i in range(0, N_GEN, 100):
        z = torch.randn(min(100, N_GEN - i), NZ, 1, 1, device=device)
        imgs = (netG(z) + 1) / 2  # [-1,1] -> [0,1]
        for j, img in enumerate(imgs):
            save_image(img, os.path.join(OUT_DIR, f"{i+j:05d}.png"))

zip_path = os.path.join(WORK_DIR, "generated.zip")
with zipfile.ZipFile(zip_path, "w") as zf:
    for p in glob.glob(os.path.join(OUT_DIR, "*.png")): zf.write(p, os.path.basename(p))
print("Saved:", zip_path, "| images:", len(glob.glob(os.path.join(OUT_DIR, "*.png"))))

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("자동 다운로드 불가:", e, "| 위치:", zip_path)

## 🏁 제출 안내

이 노트북은 **cat & dog 이미지 생성(GAN)** 데모입니다 — 생성 과제는 제출 리더보드가 없습니다.

- 학습 데이터 출처: **[Dogs vs Cats Redux](https://www.kaggle.com/c/dogs-vs-cats-redux-kernels-edition)** (고양이·강아지 사진)
- 생성 결과는 위에서 `generated.zip` 으로 저장·다운로드됩니다.

> 참고: 원래 GAN 대표 대회 *Generative Dog Images* 는 2019년 종료되어 가입/제출이 불가합니다.